# batchnorm-affine-params — ex1: apply BatchNorm's affine step to a normalized tensor

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `batchnorm-affine-params`. Running the final beacon cell reports progress against the `CNN: BatchNorm affine params` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm affine params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-affine-params`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-affine-params"
DD_SUBTOPIC = "CNN: BatchNorm affine params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BatchNorm affine params (`gamma * x̂ + beta`) — quick refresher

BatchNorm2d's forward has two stages:

```
x_hat = (x - mean) / sqrt(var + eps)        # normalize: zero-mean unit-var
y     = gamma * x_hat + beta                # affine: learnable rescale + shift
```

**Why the affine step exists.** Pure normalization would force every channel to mean-0/var-1, which removes the network's ability to learn channel-specific scale or bias. The affine params restore that capacity: `gamma` (a.k.a. `weight`) lets each channel rescale; `beta` (a.k.a. `bias`) lets each channel shift. They're learnable `nn.Parameter`s.

**Initialization convention.** `gamma` initialized to **ones**, `beta` to **zeros**. That way the layer is the **identity** at step 0 — it can't hurt training and only helps once gradient signal accumulates.

**Shape.** For a `(B, C, H, W)` input, both `gamma` and `beta` are 1-D tensors of length `C`. They must be reshaped to `(1, C, 1, 1)` before the multiply/add so they broadcast across batch and spatial axes.

**Contrast with RMSNorm.** RMSNorm has only the `gamma` scale — no `beta`. That's the architectural difference at the affine step: BatchNorm = scale + shift; RMSNorm = scale only.

### Exercise 1 — apply BatchNorm's affine step to a normalized tensor

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `y = gamma * x_hat + beta` affine step of BatchNorm2d across a `(B, C, H, W)` normalized tensor by reshaping per-channel params `(C,)` to broadcast over batch and spatial axes.
> Keywords: batchnorm, affine, gamma-beta, per-channel
> ```

**KCs targeted:** `bn-affine-formula`, `bn-per-channel-broadcast`

Implement `ex1_bn_affine(x_hat, gamma, beta)`. Given:

- `x_hat` of shape `(B, C, H, W)` — already-normalized input (mean 0, var 1 per channel).
- `gamma` of shape `(C,)` — per-channel scale (a.k.a. `weight`).
- `beta`  of shape `(C,)` — per-channel shift (a.k.a. `bias`).

Compute and return the affine output:

```
y = gamma * x_hat + beta
```

**The catch.** `gamma` and `beta` are `(C,)` — to multiply/add against a `(B, C, H, W)` tensor you must reshape them so the `C` axis lines up. The canonical reshape is `(1, C, 1, 1)`:

```
gamma.view(1, -1, 1, 1)            # or .reshape(1, C, 1, 1)
einops.rearrange(gamma, 'c -> 1 c 1 1')
```

Either form is fine. The point is that the per-channel parameter must broadcast across B, H, W — `(C,)` would incorrectly try to broadcast against the *last* axis (`W`).

**Boundary cases.**
- `gamma = ones, beta = zeros` → output equals input (the identity init that BatchNorm uses at step 0).
- `gamma = zeros` → output equals `beta` broadcast everywhere (channel becomes a constant).

In [ ]:
def ex1_bn_affine(x_hat: Tensor, gamma: Tensor, beta: Tensor) -> Tensor:
    g = gamma.view(1, -1, 1, 1)
    b = beta.view(1, -1, 1, 1)
    return g * x_hat + b


<details><summary>Solution</summary>

```python
def ex1_bn_affine(x_hat: Tensor, gamma: Tensor, beta: Tensor) -> Tensor:
    g = gamma.view(1, -1, 1, 1)
    b = beta.view(1, -1, 1, 1)
    return g * x_hat + b
```

**Why `.view(1, -1, 1, 1)`.** Adds three size-1 axes around the `C` axis so PyTorch's broadcasting matches it against `x_hat`'s channel position. Without the reshape, `(C,)` would broadcast against `x_hat`'s last axis (`W`) — silently wrong.

**Equivalent rewrites.**
- `einops.rearrange(gamma, 'c -> 1 c 1 1')` — same result, more self-documenting.
- `gamma[None, :, None, None]` — None-indexing inserts size-1 axes; works identically but is less readable.
- `gamma.reshape(1, -1, 1, 1)` — semantically identical to `.view` here because `gamma` is already contiguous.

**Why this is its own atom.** BatchNorm, LayerNorm, GroupNorm, RMSNorm, and InstanceNorm all share this final affine step. The DIFFERENCE between these layers is the **normalization** stage — which axes they reduce over to compute mean/var. The affine stage is shared. Drilling it in isolation lets you compose any norm variant by swapping the upstream normalization while reusing this exact affine code.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()